In [23]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [24]:
df = pd.read_csv('/content/Recommend.csv',names=['UserID','MovieID','Rating','TimeStamp'])
df.head()

,UserID,MovieID,Rating,TimeStamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [25]:
n_users = df.UserID.unique().shape[0]
n_movies = df.MovieID.unique().shape[0]
train_data, test_data = train_test_split(df, test_size=0.25)
n_users, n_movies

(943, 1682)

In [26]:
train_matrix = np.zeros((n_users, n_movies))
for line in train_data.itertuples():
    train_matrix[line[1]-1, line[2]-1] = line[3]
train_matrix

array([[5., 3., 4., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [27]:
test_matrix = np.zeros((n_users, n_movies))
for line in test_data.itertuples():
    test_matrix[line[1]-1, line[2]-1] = line[3]


In [28]:
from sklearn.metrics.pairwise import cosine_similarity
user_similarity = cosine_similarity(train_matrix)
movie_similarity = cosine_similarity(train_matrix.T)

In [29]:
def predict_user_based(ratings, similarity):
    mean_user_rating = ratings.mean(axis=1)[:, np.newaxis]
    ratings_diff = (ratings - mean_user_rating)
    pred = mean_user_rating + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    return pred

def predict_item_based(ratings, similarity):
    return ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])

In [30]:
user_pred = predict_user_based(train_matrix, user_similarity)
movie_pred = predict_item_based(train_matrix, movie_similarity)

/tmp/ipykernel_7950/255884874.py:8: RuntimeWarning: invalid value encountered in divide
  return ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])


In [21]:
from math import sqrt
def rmse(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten()
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    prediction[np.isnan(prediction)] = 0
    return sqrt(mean_squared_error(prediction, ground_truth))

In [22]:
print(f"User-Based CF RMSE: {rmse(user_pred, test_matrix):.4f}")
print(f"Movie-Based CF RMSE: {rmse(movie_pred, test_matrix):.4f}")

User-Based CF RMSE: 2.9582
Movie-Based CF RMSE: 3.1590
